# Làm sạch dữ liệu Khmer trên Colab / Kaggle

Notebook này chạy **các bước nặng cần GPU** trên đám mây:
1. tính embedding CLIP (`1_embed_clip.py`),
2. tìm ảnh nghi là rác (`2_find_outliers.py`, cần để lập danh sách cho Ollama),
3. (tuỳ chọn) nhận diện trước bằng Ollama (`6_ollama_prelabel.py`), T4 16 GB chạy được mô hình 7B đến 8B.

Phần **xem tay contact sheet và áp dụng kết quả** (bước 3, 4, 5) làm trên máy bạn, vì cần ảnh ở máy bạn và cần bạn nhìn ảnh. Kết quả của notebook là vài file nhỏ (`clip_embeddings.npz`, các file CSV) để tải về máy.

**Chuẩn bị trên máy bạn (PowerShell, đứng ở thư mục dự án):**
```
tar -a -cf images.zip -C khmer_dataset_v2 images
tar -a -cf clean_tools.zip clean_tools
```
(hoặc nén bằng 7-Zip, miễn là `images.zip` chứa thư mục `images/` ở gốc, và `clean_tools.zip` chứa thư mục `clean_tools/` ở gốc). Không đổi tên, xoá hay di chuyển ảnh trên máy cho đến khi tải kết quả về.

**Colab:** tải 2 file zip lên Google Drive vào thư mục `MyDrive/khmer`, chọn Runtime > Change runtime type > T4 GPU.
**Kaggle:** tạo Dataset chứa 2 file zip; trong notebook bật Accelerator = GPU T4, bật Internet (cần xác minh số điện thoại), thêm Dataset vào Input.

## Tạm dừng và chạy tiếp
- **Dừng:** bấm nút ■ (Interrupt execution) của cell đang chạy, hoặc Runtime > Interrupt execution. Đợi dòng "Đã lưu ... embedding" hoặc "Đã dừng theo yêu cầu" xuất hiện (vài giây) rồi mới đóng tab. Embedding lưu thêm tự động sau mỗi 2000 ảnh, kết quả Ollama lưu sau từng ảnh, đều ghi thẳng lên Drive.
- **Chạy tiếp (mở lại trình duyệt):** mở lại notebook, chạy lần lượt cell cấu hình, cell lấy dữ liệu (đã có in tiến độ), cell GPU, rồi chạy lại cell của bước đang dở: nó tự bỏ qua phần đã xong. Nếu dùng Ollama thì chạy lại cell cài Ollama (6a) vì máy ảo mới chưa có.
- Đừng dùng Runtime > Disconnect and delete runtime trước khi thấy dòng "Đã lưu".

In [ ]:
import os, sys, subprocess, shutil, zipfile, time
from pathlib import Path

IN_KAGGLE = Path("/kaggle").exists()
IN_COLAB = (not IN_KAGGLE) and Path("/content").exists()

# ============ CHỈNH Ở ĐÂY ============
DRIVE_DIR     = "/content/drive/MyDrive/khmer"    # Colab: thư mục Drive chứa images.zip và clean_tools.zip
CLIP_MODEL    = "openai/clip-vit-large-patch14"   # dùng cùng một mô hình CLIP cho TOÀN BỘ embedding (nhanh hơn: openai/clip-vit-base-patch32)
OLLAMA_MODEL  = "qwen3-vl:8b-instruct"            # T4 16GB chạy được 7B-8B; kiểm tra tên tại ollama.com/library
OLLAMA_LIMIT  = 20                                # chạy thử 20 ảnh trước; đặt 0 để chạy hết
# =====================================

WORK = Path("/kaggle/working/work" if IN_KAGGLE else "/content/work")
OUT_DIR = Path("/kaggle/working/results" if IN_KAGGLE else DRIVE_DIR + "/results")
WORKERS = min(4, os.cpu_count() or 2)
RESULT_FILES = ["review_list.csv", "near_duplicates.csv", "ollama_flags.csv", "unreadable_images.txt"]
LIVE_FILES = ["clip_embeddings.npz", "ollama_predictions.jsonl"]   # ghi THẲNG vào OUT_DIR (Drive) nên không mất khi phiên bị ngắt

import signal

def sh(cmd, check=True):
    """Chạy lệnh shell, in kết quả theo thời gian thực.
    Khi bạn bấm dừng (Interrupt), lệnh con nhận đúng MỘT tín hiệu dừng và có thời gian lưu kết quả."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         start_new_session=True)   # tách phiên để chỉ nhận tín hiệu dừng do mình gửi
    assert p.stdout is not None
    try:
        for line in p.stdout:
            print(line, end="")
    except KeyboardInterrupt:
        print("\n[Đang dừng, chờ script lưu kết quả (tối đa 2 phút)...]")
        for child in subprocess.run(["pgrep", "-P", str(p.pid)], capture_output=True, text=True).stdout.split():
            try: os.kill(int(child), signal.SIGINT)   # shell bọc ngoài -> gửi cho python bên trong
            except ProcessLookupError: pass
        try:
            for line in p.stdout:
                print(line, end="")
            p.wait(timeout=120)
        except Exception:
            p.kill()
        raise
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"Lệnh lỗi (mã {rc}): {cmd}")
    return rc

print("Môi trường:", "Kaggle" if IN_KAGGLE else "Colab" if IN_COLAB else "khác (máy cục bộ)")

In [ ]:
# Cell 2: lấy dữ liệu về ổ đĩa của máy ảo và giải nén
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SRC = Path(DRIVE_DIR)
else:
    SRC = Path("/kaggle/input")

def find_zip(name):
    hits = sorted(SRC.rglob(name))
    if not hits:
        raise FileNotFoundError(f"Không thấy {name} trong {SRC}. Kiểm tra lại bước chuẩn bị ở đầu notebook.")
    return hits[0]

WORK.mkdir(parents=True, exist_ok=True)
ROOT, TOOLS = WORK / "khmer_dataset_v2", WORK / "clean_tools"
if not (ROOT / "images").is_dir():
    ROOT.mkdir(parents=True, exist_ok=True)
    print("Giải nén ảnh (vài phút)...")
    with zipfile.ZipFile(find_zip("images.zip")) as z:
        z.extractall(ROOT)
if not (TOOLS / "1_embed_clip.py").exists():
    with zipfile.ZipFile(find_zip("clean_tools.zip")) as z:
        z.extractall(WORK)

assert (ROOT / "images").is_dir(), "images.zip phải chứa thư mục images/ ở gốc"
assert (TOOLS / "1_embed_clip.py").exists(), "clean_tools.zip phải chứa thư mục clean_tools/ ở gốc"
n_img = sum(1 for _ in (ROOT / "images").rglob("*") if _.is_file())
print(f"Sẵn sàng: {n_img} file ảnh tại {ROOT}")

# Khôi phục kết quả đã lưu từ lần chạy trước (để chạy tiếp sau khi máy ảo bị ngắt)
OUT_DIR.mkdir(parents=True, exist_ok=True)
for f in RESULT_FILES + ["clip_embeddings.npz"]:
    if (OUT_DIR / f).exists() and not (ROOT / f).exists():
        shutil.copy2(OUT_DIR / f, ROOT / f)
        print("Khôi phục:", f)

def backup():
    """Sao các file kết quả (nhỏ) ra nơi lưu bền: Drive (Colab) hoặc /kaggle/working/results."""
    for f in RESULT_FILES:
        if (ROOT / f).exists():
            shutil.copy2(ROOT / f, OUT_DIR / f)
    print("Đã lưu kết quả vào", OUT_DIR)

def status():
    """Cho biết đang ở đâu để biết cần chạy tiếp từ cell nào."""
    import numpy as np
    total = sum(1 for _ in (ROOT / "images").rglob("*") if _.is_file())
    emb = OUT_DIR / "clip_embeddings.npz"
    n_emb = len(np.load(emb)["names"]) if emb.exists() else 0
    print(f"Bước 1 (embedding): {n_emb}/{total} ảnh", "-> XONG" if n_emb >= total else "-> chạy cell Bước 1 để tiếp tục")
    rl = ROOT / "review_list.csv"
    n_rl = max(0, sum(1 for _ in open(rl, encoding="utf-8-sig")) - 1) if rl.exists() else 0
    print(f"Bước 2 (danh sách nghi rác): {n_rl} ảnh", "-> XONG" if n_rl else "-> chưa chạy")
    pj = OUT_DIR / "ollama_predictions.jsonl"
    n_pj = sum(1 for _ in open(pj, encoding="utf-8")) if pj.exists() else 0
    print(f"Bước 6 (Ollama): {n_pj}/{n_rl} ảnh đã nhận diện" if n_rl else "Bước 6 (Ollama): chưa có danh sách")

status()

In [ ]:
# Cell 3: kiểm tra GPU và cài thư viện
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", check=False)
sh("pip install -q transformers scikit-learn pillow numpy requests")
import torch
print("CUDA:", torch.cuda.is_available(), "| torch", torch.__version__)
assert torch.cuda.is_available(), "Chưa bật GPU: vào Runtime/Accelerator và chọn T4 GPU"

## Bước 1: tính embedding CLIP (chạy tiếp được nếu bị ngắt)

In [ ]:
CACHE = OUT_DIR / "clip_embeddings.npz"      # ghi thẳng lên Drive, lưu định kỳ sau mỗi 2000 ảnh
try:
    sh(f"cd {TOOLS} && python 1_embed_clip.py --root {ROOT} --cache {CACHE} --model {CLIP_MODEL} "
       f"--batch 128 --workers {WORKERS} --save-every 2000")
finally:
    if CACHE.exists():
        shutil.copy2(CACHE, ROOT / "clip_embeddings.npz")   # các bước sau đọc bản này
    status()

## Bước 2: tìm ảnh nghi là rác
(Trên đám mây không cần sao thư mục `centroid_samples`; bạn chạy lại bước này ở máy để có thư mục đó.)

In [ ]:
sh(f"cd {TOOLS} && python 2_find_outliers.py --root {ROOT} --centroid-samples 0")
backup()

## Bước 6 (tuỳ chọn): Ollama nhận diện trước
Chỉ chạy các ảnh trong `review_list.csv`. Chạy thử `OLLAMA_LIMIT = 20` trước, xem chất lượng rồi đặt `OLLAMA_LIMIT = 0` để chạy hết. Có thể chạy lại cell cuối nhiều lần, ảnh đã xong sẽ được bỏ qua; kết quả luôn được lưu khi cell kết thúc hoặc bị dừng.

In [ ]:
# Cell 6a: cài Ollama và khởi động server (cần Internet)
sh("apt-get update -qq && apt-get install -y -qq zstd pciutils curl")
sh("curl -fsSL https://ollama.com/install.sh | sh")
ollama_log = open("/tmp/ollama.log", "w")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)
import urllib.request
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Ollama không khởi động được, xem /tmp/ollama.log")
sh(f"ollama pull {OLLAMA_MODEL}")

In [ ]:
# Cell 6b: chạy nhận diện (chạy lại được nhiều lần)
limit = f"--limit {OLLAMA_LIMIT}" if OLLAMA_LIMIT else ""
try:
    sh(f"cd {TOOLS} && python 6_ollama_prelabel.py --root {ROOT} --model {OLLAMA_MODEL} "
       f"--only-list {ROOT}/review_list.csv --predictions {OUT_DIR}/ollama_predictions.jsonl {limit}")
finally:
    backup()
    status()

## Đóng gói kết quả để tải về

In [ ]:
zip_path = OUT_DIR / "khmer_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in RESULT_FILES + LIVE_FILES:
        src = (OUT_DIR if f in LIVE_FILES else ROOT) / f
        if src.exists():
            z.write(src, f)
print("Đã tạo:", zip_path, f"({zip_path.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print("Kaggle: sau khi chạy xong (Save Version > Save & Run All), tải file này ở mục Output của notebook.")

## Làm tiếp trên máy của bạn

Giải nén `khmer_results.zip` vào thư mục `khmer_dataset_v2` (cạnh thư mục `images`), rồi chạy ở PowerShell:

```
cd clean_tools
python 2_find_outliers.py --root ..\khmer_dataset_v2     # tạo thêm thư mục centroid_samples, chạy bằng CPU vài giây
python 3_contact_sheets.py --root ..\khmer_dataset_v2
```
Nếu đã chạy Ollama, xem thêm các ảnh bị gắn cờ:
```
python 3_contact_sheets.py --root ..\khmer_dataset_v2 --list ..\khmer_dataset_v2\ollama_flags.csv --out ..\khmer_dataset_v2\sheets_ollama
```
Sau đó xem tay, `4_apply_review.py`, `5_probe_check.py` như đã hướng dẫn.